# Coverage, budgets, and deadlines

`report.coverage` records what each detector examined, at what scale,
and what was skipped or capped and why. Read it alongside findings to
understand which properties were checked. This chapter explains the coverage
vocabulary, the three budget presets, the optional wall-clock deadline, the
overrides that scale the sampled work, and how detector selection appears in
coverage.

## The coverage vocabulary

`report.coverage` maps each pass to a status and a human-readable detail
line. There are five statuses:

| Status | Meaning | Typical detail |
|---|---|---|
| `ran_exact` | The pass ran over all rows, no sampling | "6 cards (exact)" |
| `ran_sampled` | The pass ran on a deterministic sample by design | sample size vs table size |
| `capped` | The pass ran but a search cap limited its scope | how many of how many candidates |
| `skipped` | The pass did not run at all | "quick tier", "deadline reached", "detector not enabled" |
| `not_applicable` | The pass ran and found nothing to report on | "0 single-column unique, 0 composite" |

The distinction that matters most is `capped` versus `skipped`: a capped
pass still screened everything and only bounded the expensive follow-up
(for example, all numeric pairs were screened, and the strongest 496 got
detailed statistics); a skipped pass did nothing, and absence of its
findings means nothing.

## The three budgets

The `budget` argument selects one of three presets. Everything exact is
identical across budgets; what changes is the relationship work.

| | `quick` | `standard` | `deep` |
|---|---|---|---|
| Column cards, duplicates, keys, sortedness, dates | all exact, full rows | same | same |
| Single-column uniqueness | exact | exact | exact |
| Composite key search | — | pairs, verified up to 16 | pairs and triples, verified up to 24 |
| Functional dependencies | — | LHS size ≤ 2, verified over ≤ 200,000 rows | LHS size ≤ 3, verified over ≤ 500,000 rows |
| Model passes (screening, tables, rules, regimes) | — | ≤ 20,000 encoded rows | ≤ 50,000 encoded rows |
| Numeric pair statistics | — | ≤ 496 pairs over ≤ 20,000 rows | ≤ 2,016 pairs over ≤ 50,000 rows |

`quick` runs column summaries and basic integrity checks. `standard`
adds relationship discovery and is the default. `deep` uses wider
searches and larger samples. The exact
numbers behind these presets, and the reasoning, are in
[](../methodology/sampling.ipynb).

In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(3)
n = 15_000

region_id = [rng.randrange(4) for _ in range(n)]
region_name = [["north", "south", "east", "west"][r] for r in region_id]
quantity = [rng.randrange(1, 21) for _ in range(n)]
double_quantity = [2 * q for q in quantity]
x = [rng.uniform(-1, 1) for _ in range(n)]
y = [v * v for v in x]

df = pl.DataFrame({
    "order_id": list(range(n)),
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
    "x": x,
    "y": y,
})

reports = {b: onceover.profile(df, budget=b) for b in ["quick", "standard", "deep"]}
for b, r in reports.items():
    measured = sum(1 for f in r.findings
                   if any(e["basis"] == "measured" for e in f["evidence"]))
    print(f"{b:9s} {len(r.findings):3d} findings ({measured} with measured evidence)")

The same table at three budgets: `quick` returns full-data summaries
and basic integrity findings; `standard` adds relationships; `deep` finds the same
structure here with wider search — this fixture's relationships are simple
enough to surface at `standard`.

Pass-by-pass, the statuses make the differences explicit:

In [ ]:
from great_tables import GT, loc, style

passes = sorted(set().union(*[set(r.coverage) for r in reports.values()]))
rows = []
for p in passes:
    row = {"pass": p}
    for b, r in reports.items():
        row[b] = r.coverage.get(p, {"status": "—", "detail": ""})["status"]
    rows.append(row)

(GT(pl.DataFrame(rows))
 .tab_header(title="Pass status by budget", subtitle="15,000 rows × 7 columns")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"),
            loc.body()))

## Overrides

Three keyword arguments scale the sampled work without touching the exact
passes. `sample_rows` caps the row sample shared by the FD verification, the
encoded matrix, and model fitting. `max_predictors` caps how many predictors
survive screening per fitted model. `max_pairs` caps how many numeric pairs
get detailed statistics after screening. Column cards, duplicate detection,
single-column keys, sortedness, dates, and identity residual verification
always examine the full input.

In [ ]:
default = onceover.profile(df)
tuned = onceover.profile(df, sample_rows=6_000, max_predictors=8, max_pairs=20)

for pass_name in ["encoding", "functional_dependencies", "dependence_screen"]:
    print(f"{pass_name}:")
    print("  default:", default.coverage[pass_name]["detail"][:90])
    print("  tuned:  ", tuned.coverage[pass_name]["detail"][:90])

The tuned run's coverage shows 6,000-row samples where the default used
15,000 (and would use 20,000 on a larger table). Smaller samples are
cheaper but can leave too few rows for model fitting; the coverage detail
is where that compromise becomes visible. Absence of a finding after an
override means absence at that sample size — nothing stronger.

## The deadline

An optional wall-clock budget, `deadline_seconds`, trades completeness for
time. It is checked *between* passes: whichever pass is running when the
deadline passes still completes, and later passes are skipped and recorded
as `"deadline reached"` in coverage. Nothing is interrupted mid-computation,
so a finding that was emitted is still a fully computed finding.

The deadline is the single opt-in source of nondeterminism in the package.
Without it, the same data and configuration produce byte-identical reports;
with it, which passes finish depends on machine speed at that moment — and
the coverage block records exactly what was skipped, so readers can see which passes completed in that run.

In [ ]:
import numpy as np

rng_np = np.random.default_rng(5)
big = pl.DataFrame(
    {"id": np.arange(400_000, dtype=np.int64)}
    | {f"x{i:02}": rng_np.random(400_000) for i in range(15)}
)

timed = onceover.profile(big, deadline_seconds=1.0)
for name, info in timed.coverage.items():
    if info["status"] == "skipped":
        print(f"{name:28s} {info['status']:14s} {info['detail']}")

On a 400,000-row frame the exact passes finish quickly; the deadline
catches the model-based passes. The passes skipped depend on machine speed. Include the coverage block
when sharing a report produced under a deadline; it records which passes
completed.

## Detector selection

Findings can be composed by choosing detectors, and coverage is where the
choice becomes visible. Selecting a subset with `detectors=[...]` marks the
deselected passes `"detector not enabled"`; a detector that another
requires still runs, but its coverage line gains "required as a resource;
findings not selected". And every rule candidate — from PRISM or from
association mining — is re-counted on a common diagnostic partition, whose
result appears as the `rule_evaluation` pass. The resource semantics and
examples are in [](findings.ipynb); coverage records which passes ran and why.

## Reading coverage in practice

Three questions cover most situations:

- **"Why is there no finding of this kind?"** — find the pass in coverage.
  `skipped` and `capped` mean the question was not fully asked; `ran_exact`
  with no finding means the pass ran but reported no property that met
  its criteria.
- **"How big was the sample?"** — the detail line always pairs sample size
  with table size, and each finding's evidence `scope` records
  `population_rows` versus `rows_evaluated` individually.
- **"Can I trust this enough to automate on it?"** — combine the finding's
  evidence (basis, sampling, validation) with the pass status and the
  intended use. Full-data measurements describe the current table;
  sampled and model-based findings need validation on the data where
  an automated action will apply.